In [1]:
!pip install matplotlib pandas seaborn numpy nltk

In [1]:
%matplotlib inline

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import _pickle as cPickle
import nltk

In [24]:
df = pd.read_csv('lsapp.tsv', sep='\t')
df_open = df.loc[df['event_type'] == 'Opened']

In [25]:
df_open.loc[:,'timestamp'] = pd.to_datetime(df_open['timestamp'])
df.loc[:,'timestamp'] = pd.to_datetime(df['timestamp'])

In [26]:
len(df['session_id'].unique())

76247

In [27]:
df

,user_id,session_id,timestamp,app_name,event_type
0,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Closed
2,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Opened
3,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Closed
4,0,1,2018-01-16 06:01:08,Minesweeper Classic (Mines),Opened
...,...,...,...,...,...
3658584,291,76247,2018-04-06 14:35:15,Facebook,Closed
3658585,291,76247,2018-04-06 14:35:15,Facebook,Opened
3658586,291,76247,2018-04-06 14:35:37,Facebook,Closed
3658587,291,76247,2018-04-06 14:35:37,Facebook,Opened


In [19]:
df['time_dff']  = df[['timestamp']].diff()
df['interaction_id'] = (((df.timestamp-df.timestamp.shift(1) > pd.Timedelta(1, 'm')) 
                              & (df.event_type == 'Opened')) 
                              | ~(df.app_name == df.app_name.shift(1))
                              | ~(df.user_id == df.user_id.shift(1))).cumsum()
# df['session_id'] = (((df.timestamp-df.timestamp.shift(1) > pd.Timedelta(5, 'm')) 
#                               & (df.event_type == 'Opened')) 
#                               | ~(df.user_id == df.user_id.shift(1))).cumsum()
# df_start = df.drop_duplicates(subset=['interaction_id'], keep='first')
df_start = df
# df_end = df.drop_duplicates(subset=['interaction_id'], keep='last')
df_end = df
df_start.set_index('interaction_id', inplace=True)
df_end.set_index('interaction_id', inplace=True)
df_start.loc[:, 'open_time'] = df_start['timestamp']
df_start.loc[:, 'close_time']  = df_end['timestamp']

In [20]:
len(df['session_id'].unique())

76247

In [21]:
df_start

,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
interaction_id,,,,,,,,
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Closed,0 days 00:00:00,2018-01-16 06:01:05,2018-01-16 06:01:09
1,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Opened,0 days 00:00:02,2018-01-16 06:01:07,2018-01-16 06:01:09
1,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Closed,0 days 00:00:00,2018-01-16 06:01:07,2018-01-16 06:01:09
1,0,1,2018-01-16 06:01:08,Minesweeper Classic (Mines),Opened,0 days 00:00:01,2018-01-16 06:01:08,2018-01-16 06:01:09
...,...,...,...,...,...,...,...,...
599634,291,76247,2018-04-06 14:35:15,Facebook,Closed,0 days 00:00:41,2018-04-06 14:35:15,2018-04-06 14:35:37
599634,291,76247,2018-04-06 14:35:15,Facebook,Opened,0 days 00:00:00,2018-04-06 14:35:15,2018-04-06 14:35:37
599634,291,76247,2018-04-06 14:35:37,Facebook,Closed,0 days 00:00:22,2018-04-06 14:35:37,2018-04-06 14:35:37


In [22]:
len(df_start['session_id'].unique())

76247

In [23]:
df_start.to_csv('df_start.tsv')